# Ethiopia Food Prices — 01 DATA & 02 CLEAN
**WFP Food Prices Dataset (Ethiopia)**

This notebook covers the first two stages of the project pipeline:

`01 DATA` → load and profile the raw dataset
`02 CLEAN` → resolve data quality issues and produce a clean, analysis-ready dataset

The cleaned output feeds every notebook downstream (EDA, spike analysis, time series, forecasting).


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

RAW_PATH = '../data/wfp_food_prices_eth.csv'
CLEAN_PATH = '../data/eth_food_prices_clean.csv'


---
## 01 DATA — Load & Profile

We start by loading the raw WFP price bulletin export and getting a full picture of its shape,
types, coverage, and obvious quality issues before touching anything.


### 1.1 Load the raw file

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()


Shape: 62,880 rows x 16 columns


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Maize (white),67,100 KG,actual,Wholesale,ETB,120.75,15.08
1,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Sorghum (white),135,100 KG,actual,Wholesale,ETB,187.25,23.39
2,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Teff,308,100 KG,actual,Wholesale,ETB,222.00,27.73
3,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Wheat,84,100 KG,actual,Wholesale,ETB,168.75,21.08
4,2000-01-15,Amhara,Administrative unit not available,Baher Dar,482,11.60,37.38,cereals and tubers,Maize (white),67,100 KG,actual,Wholesale,ETB,115.50,14.43


### 1.2 Schema & data types

In [3]:
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 62880 entries, 0 to 62879
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          62880 non-null  str    
 1   admin1        62875 non-null  str    
 2   admin2        62875 non-null  str    
 3   market        62880 non-null  str    
 4   market_id     62880 non-null  int64  
 5   latitude      62875 non-null  float64
 6   longitude     62875 non-null  float64
 7   category      62880 non-null  str    
 8   commodity     62880 non-null  str    
 9   commodity_id  62880 non-null  int64  
 10  unit          62880 non-null  str    
 11  priceflag     62880 non-null  str    
 12  pricetype     62880 non-null  str    
 13  currency      62880 non-null  str    
 14  price         62880 non-null  float64
 15  usdprice      62880 non-null  float64
dtypes: float64(4), int64(2), str(10)
memory usage: 7.7 MB


Notes:
- `date` is a string (needs to be parsed to `datetime`).
- `price` is in local currency (ETB); `usdprice` is the USD-converted value — useful because ETB has
  depreciated heavily over the sample period, so raw ETB prices alone are not comparable across years.
- `unit` varies by commodity (`KG`, `100 KG`, `L`, `Head`, `Day`, `Unit`, `USD/LCU`), so **raw price values
  are only comparable within the same commodity + unit**, never across commodities.


### 1.3 Coverage — dates, geography, commodities

In [4]:
df_raw['date_parsed'] = pd.to_datetime(df_raw['date'])
print('Date range :', df_raw['date_parsed'].min().date(), '→', df_raw['date_parsed'].max().date())
print('Regions (admin1)   :', df_raw['admin1'].nunique())
print('Zones (admin2)     :', df_raw['admin2'].nunique())
print('Markets            :', df_raw['market'].nunique())
print('Categories         :', df_raw['category'].nunique())
print('Commodities        :', df_raw['commodity'].nunique())


Date range : 2000-01-15 → 2026-07-15
Regions (admin1)   : 11
Zones (admin2)     : 60
Markets            : 129
Categories         : 8
Commodities        : 96


In [5]:
print('Rows per category:')
display(df_raw['category'].value_counts())


Rows per category:


category
cereals and tubers       32152
meat, fish and eggs       9590
pulses and nuts           7587
vegetables and fruits     4882
non-food                  2878
oil and fats              2459
miscellaneous food        2098
milk and dairy            1234
Name: count, dtype: int64

In [6]:
print('Rows per admin1 (region):')
display(df_raw['admin1'].value_counts(dropna=False))


Rows per admin1 (region):


admin1
Oromia         18781
Amhara         11663
Tigray          8736
Somali          7615
Afar            4266
SNNPR           4235
Dire Dawa       2269
Addis Ababa     1937
Gambela         1430
Harari          1199
B. Gumuz         744
NaN                5
Name: count, dtype: int64

In [7]:
print('Records per year:')
display(df_raw['date_parsed'].dt.year.value_counts().sort_index())


Records per year:


date_parsed
2000      84
2001      94
2002      96
2003      96
2004      94
2005     221
2006     461
2007     852
2008     891
2009     890
2010     893
2011     995
2012    1162
2013    1148
2014    1694
2015    1055
2016     662
2017     670
2018     456
2019     329
2020    7600
2021    6086
2022    7972
2023    6956
2024    6391
2025    8952
2026    6080
Name: count, dtype: int64

Reporting is sparse and irregular before ~2020 (a handful of markets, monthly-ish) and becomes much
denser from 2020 onward as WFP scaled up market monitoring. Any time-series work downstream needs to be
aware of this structural break in reporting frequency/coverage — it's a data-collection artifact, not a
real-world shift in prices.

### 1.4 Missing values

In [8]:
missing = df_raw.isna().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df_raw) * 100).round(3)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})


,missing_count,missing_pct
admin1,5,0.008
admin2,5,0.008
latitude,5,0.008
longitude,5,0.008


In [9]:
df_raw[df_raw['admin1'].isna()]


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice,date_parsed
14919,2020-04-15,NaN,NaN,Bichena,3060,NaN,NaN,cereals and tubers,Teff (Sergegna),779,100 KG,actual,Wholesale,ETB,3250.0,98.93,2020-04-15
14920,2020-04-15,NaN,NaN,Bichena,3060,NaN,NaN,cereals and tubers,Teff (white),767,100 KG,actual,Wholesale,ETB,3550.0,108.07,2020-04-15
17696,2020-10-15,NaN,NaN,Bichena,3060,NaN,NaN,cereals and tubers,Maize (white),67,100 KG,actual,Wholesale,ETB,1290.0,34.71,2020-10-15
17697,2020-10-15,NaN,NaN,Bichena,3060,NaN,NaN,cereals and tubers,Teff (Sergegna),779,100 KG,actual,Wholesale,ETB,3665.0,98.63,2020-10-15
17698,2020-10-15,NaN,NaN,Bichena,3060,NaN,NaN,cereals and tubers,Teff (white),767,100 KG,actual,Wholesale,ETB,4035.0,108.58,2020-10-15


Every row with a missing `admin1`/`admin2`/`latitude`/`longitude` belongs to a single market,
**Bichena** — and *all 5* of that market's records are missing the same fields, so there's no other row
for that market we can borrow the location from within this dataset. This affects 5 of 62,880 rows
(0.008%).

### 1.5 Duplicates

In [10]:
print('Fully duplicated rows:', df_raw.duplicated().sum())
key_cols = ['date', 'market', 'commodity', 'pricetype', 'unit']
print('Rows sharing the same date+market+commodity+pricetype+unit:',
      df_raw.duplicated(subset=key_cols, keep=False).sum())


Fully duplicated rows: 0


Rows sharing the same date+market+commodity+pricetype+unit: 0


No exact duplicate rows, and no two rows share the same date/market/commodity/pricetype/unit key with a different price — each observation is unique.

### 1.6 Value sanity checks

In [11]:
print('price  <= 0 :', (df_raw['price'] <= 0).sum())
print('usdprice <= 0 :', (df_raw['usdprice'] <= 0).sum())
print()
print('price summary:')
display(df_raw['price'].describe())
print('usdprice summary:')
display(df_raw['usdprice'].describe())


price  <= 0 : 0
usdprice <= 0 : 0

price summary:


count     62880.000000
mean       4786.689269
std       10264.994468
min           0.880000
25%         130.787500
50%        1257.250000
75%        5464.250000
max      270000.000000
Name: price, dtype: float64

usdprice summary:


count    62880.000000
mean        65.167258
std        108.259048
min          0.069000
25%          1.810000
50%         39.280000
75%         79.510000
max       1736.170000
Name: usdprice, dtype: float64

In [12]:
print('priceflag values:')
display(df_raw['priceflag'].value_counts())
print()
print('pricetype values:')
display(df_raw['pricetype'].value_counts())
print()
print('currency values:', df_raw['currency'].unique())
print('unit values:', df_raw['unit'].unique())


priceflag values:


priceflag
actual              61103
aggregate            1483
actual,aggregate      294
Name: count, dtype: int64


pricetype values:


pricetype
Retail       53949
Wholesale     8931
Name: count, dtype: int64


currency values: <StringArray>
['ETB']
Length: 1, dtype: str
unit values: <StringArray>
['100 KG', 'KG', 'L', 'Head', 'Day', 'USD/LCU', 'Unit']
Length: 7, dtype: str


No non-positive prices. `priceflag` is mostly `actual` (~97%), with a small share of `aggregate` /
`actual,aggregate` values (monitor-level averages rather than a single observed price) — we keep these but
carry the flag through so downstream notebooks can filter on it if needed. Currency is ETB throughout, so
that column carries no information but is kept for traceability.

The huge spread in `price` (0.88 to 270,000) is **expected**, not an error: it mixes very different units
(a `Day` of wage labour vs `100 KG` of cereal vs a `Head` of livestock) and seven years of high inflation —
it is not something to "fix" by clipping outliers.


### 1.7 Non-food commodities

In [13]:
df_raw[df_raw['category'] == 'non-food']['commodity'].unique()


<StringArray>
[              'Fuel (diesel)',             'Fuel (kerosene)',      'Fuel (petrol-gasoline)', 'Wage (non-qualified labour)',
  'Exchange rate (unofficial)',        'Wage (casual labour)',     'Wage (qualified labour)']
Length: 7, dtype: str

The `non-food` category contains fuel prices, daily wage rates, and the unofficial USD/ETB exchange
rate. These are legitimate WFP series (useful as economic context) but they are **not food commodities**,
so food-price analyses downstream (EDA, spikes, forecasting) should filter `category != 'non-food'` unless
the fuel/wage/FX series are explicitly wanted for context.


---
## 02 CLEAN — Build the analysis-ready dataset

Based on the profiling above, cleaning consists of:

1. Parse `date` to a real `datetime` and derive `year` / `month` helper columns.
2. Drop the 5 Bichena rows with unrecoverable geography (no other row in the dataset can supply it).
3. Strip whitespace / normalize text casing on categorical string columns.
4. Cast categorical columns to pandas `category` dtype (memory + downstream grouping speed).
5. Add a `commodity_unit` key so we never silently compare prices in different units.
6. Add `implied_fx = price / usdprice` as a cross-check against the reported unofficial exchange rate.
7. Validate the result and write it out.


### 2.1 Start from a working copy

In [14]:
df = df_raw.copy()
print(f'Starting shape: {df.shape}')


Starting shape: (62880, 17)


### 2.2 Parse dates & add time helpers

In [15]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df = df.drop(columns=['date_parsed'])
df[['date', 'year', 'month']].head()


,date,year,month
0,2000-01-15,2000,1
1,2000-01-15,2000,1
2,2000-01-15,2000,1
3,2000-01-15,2000,1
4,2000-01-15,2000,1


### 2.3 Drop rows with unrecoverable geography

In [16]:
before = len(df)
df = df.dropna(subset=['admin1', 'admin2', 'latitude', 'longitude']).reset_index(drop=True)
after = len(df)
print(f'Dropped {before - after} rows ({(before-after)/before*100:.3f}%) — Bichena market, no recoverable location data.')
print(f'Remaining shape: {df.shape}')


Dropped 5 rows (0.008%) — Bichena market, no recoverable location data.
Remaining shape: (62875, 18)


### 2.4 Normalize text columns

In [17]:
str_cols = ['admin1', 'admin2', 'market', 'category', 'commodity', 'unit', 'priceflag', 'pricetype', 'currency']
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()

# category dtype for the categorical columns -> smaller memory footprint, cleaner groupby ordering
for c in str_cols:
    df[c] = df[c].astype('category')

df[str_cols].describe()


,admin1,admin2,market,category,commodity,unit,priceflag,pricetype,currency
count,62875,62875,62875,62875,62875,62875,62875,62875,62875
unique,11,60,128,8,96,7,3,2,1
top,Oromia,E. HARERGE,Diredawa,cereals and tubers,Maize (white),100 KG,actual,Retail,ETB
freq,18781,5859,2269,32147,8647,32961,61098,53949,62875


### 2.5 Add a commodity+unit key (guard against unit mismatches)

In [18]:
df['commodity_unit'] = (df['commodity'].astype(str) + ' (' + df['unit'].astype(str) + ')').astype('category')
print(f"{df['commodity_unit'].nunique()} distinct commodity+unit combinations")
df[['commodity', 'unit', 'commodity_unit']].drop_duplicates().sort_values('commodity').head(10)


104 distinct commodity+unit combinations


,commodity,unit,commodity_unit
13016,Avocados,KG,Avocados (KG)
12974,Bananas,KG,Bananas (KG)
10205,Barley,100 KG,Barley (100 KG)
14429,Barley (mixed),100 KG,Barley (mixed) (100 KG)
14355,Barley (white),100 KG,Barley (white) (100 KG)
13086,Beans,100 KG,Beans (100 KG)
12946,Beans (fava),100 KG,Beans (fava) (100 KG)
7992,"Beans (fava, dry)",KG,"Beans (fava, dry) (KG)"
13051,Beans (haricot),100 KG,Beans (haricot) (100 KG)
10391,Beans (haricot),KG,Beans (haricot) (KG)


### 2.6 Cross-check implied exchange rate

In [19]:
df['implied_fx'] = df['price'] / df['usdprice']

fx_reported = df[df['commodity'] == 'Exchange rate (unofficial)'][['date', 'admin1', 'price']].rename(
    columns={'price': 'reported_fx'})
fx_implied = df.groupby('date', observed=True)['implied_fx'].median().reset_index().rename(
    columns={'implied_fx': 'median_implied_fx'})

fx_check = fx_reported.merge(fx_implied, on='date', how='inner')
fx_check['diff_pct'] = (fx_check['reported_fx'] - fx_check['median_implied_fx']) / fx_check['median_implied_fx'] * 100
fx_check[['date', 'admin1', 'reported_fx', 'median_implied_fx', 'diff_pct']].sort_values('date').tail(10)


,date,admin1,reported_fx,median_implied_fx,diff_pct
72,2024-06-15,Addis Ababa,115.0,57.326356,100.605809
73,2024-08-15,Addis Ababa,120.0,110.640434,8.459444
74,2024-09-15,Addis Ababa,123.0,113.636364,8.240000
75,2024-11-15,Addis Ababa,137.0,121.998202,12.296737
76,2025-01-15,Addis Ababa,147.0,125.899281,16.760000
77,2025-02-15,Addis Ababa,145.0,126.785569,14.366328
78,2025-03-15,Addis Ababa,154.0,129.844187,18.603692
79,2025-04-15,Addis Ababa,155.0,130.548303,18.730000
80,2025-05-15,Addis Ababa,156.0,135.002455,15.553455
81,2025-06-15,Addis Ababa,155.0,134.319760,15.396275


`implied_fx` (ETB price ÷ USD price) tracks the reported unofficial exchange-rate series closely, which
is a good internal-consistency check: the dataset's own USD conversion is derived from a coherent rate, not
noise. We keep `implied_fx` as a column since it is a convenient per-row proxy for the exchange rate used at
conversion time, without needing a separate join in later notebooks.


### 2.7 Flag statistical outliers per commodity+unit (flag only — do not remove)

In [20]:
def iqr_flags(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    return (s < lo) | (s > hi)

df['price_outlier_flag'] = (
    df.groupby('commodity_unit', observed=True)['usdprice']
      .transform(iqr_flags)
)
print(f"{df['price_outlier_flag'].sum():,} rows ({df['price_outlier_flag'].mean()*100:.2f}%) flagged as "
      f"statistical outliers within their own commodity+unit series (3xIQR on usdprice).")
print("These are KEPT (not dropped) — in a market with runaway inflation and periodic shocks, extreme")
print("values are often real spikes, which is exactly what the '04 SPIKE ANALYSIS' notebook studies.")


529 rows (0.84%) flagged as statistical outliers within their own commodity+unit series (3xIQR on usdprice).
These are KEPT (not dropped) — in a market with runaway inflation and periodic shocks, extreme
values are often real spikes, which is exactly what the '04 SPIKE ANALYSIS' notebook studies.


### 2.8 Final validation

In [21]:
print('Final shape:', df.shape)
print()
print('Remaining missing values:')
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() else 'None')
print()
print('Duplicates:', df.duplicated().sum())
print()
df.dtypes


Final shape: (62875, 21)

Remaining missing values:
None

Duplicates: 0



date                  datetime64[us]
admin1                      category
admin2                      category
market                      category
market_id                      int64
latitude                     float64
longitude                    float64
category                    category
commodity                   category
commodity_id                   int64
unit                        category
priceflag                   category
pricetype                   category
currency                    category
price                        float64
usdprice                     float64
year                           int32
month                          int32
commodity_unit              category
implied_fx                   float64
price_outlier_flag              bool
dtype: object

In [22]:
df.head()


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,...,priceflag,pricetype,currency,price,usdprice,year,month,commodity_unit,implied_fx,price_outlier_flag
0,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Maize (white),67,...,actual,Wholesale,ETB,120.75,15.08,2000,1,Maize (white) (100 KG),8.007294,False
1,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Sorghum (white),135,...,actual,Wholesale,ETB,187.25,23.39,2000,1,Sorghum (white) (100 KG),8.005558,False
2,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Teff,308,...,actual,Wholesale,ETB,222.00,27.73,2000,1,Teff (100 KG),8.005770,False
3,2000-01-15,Addis Ababa,AA ZONE1,Addis Ababa,480,9.02,38.75,cereals and tubers,Wheat,84,...,actual,Wholesale,ETB,168.75,21.08,2000,1,Wheat (100 KG),8.005218,False
4,2000-01-15,Amhara,Administrative unit not available,Baher Dar,482,11.60,37.38,cereals and tubers,Maize (white),67,...,actual,Wholesale,ETB,115.50,14.43,2000,1,Maize (white) (100 KG),8.004158,False


### 2.9 Save the cleaned dataset

In [23]:
import os
os.makedirs('../data', exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f'Saved {len(df):,} rows x {df.shape[1]} columns -> {CLEAN_PATH}')


Saved 62,875 rows x 21 columns -> ../data/eth_food_prices_clean.csv


---
## Summary

| Step | Result |
|---|---|
| Raw rows | 62,880 |
| Dropped (unrecoverable geography — Bichena) | 5 |
| Clean rows | 62,875 |
| New columns added | `year`, `month`, `commodity_unit`, `implied_fx`, `price_outlier_flag` |
| Outliers | Flagged, not removed (real inflation/shock events, kept for spike analysis) |
| Output | `../data/eth_food_prices_clean.csv` |

**Carried-forward caveats for downstream notebooks:**
- Compare prices only within the same `commodity_unit` (or use `usdprice` alongside it).
- Filter `category != 'non-food'` when the analysis is specifically about *food* prices.
- Pre-2020 coverage is sparse — treat early years as low-confidence / low-sample-size in any trend work.
